# 01 Source cluster tests

This notebook runs group-level cluster permutation tests on subject-level source estimates that were morphed to fsaverage (`3_source_modeling/05_morph_evoked_source_estimates_to_fsaverage.ipynb`). The tests are defined once under `stats.analyses` in `configs/local.yaml`; see the README section "Group statistics".

Inputs per analysis and subject:

- morphed evoked source estimates for every condition of the analysis (`source_estimates_fsaverage/`),
- for `regression` analyses a scores table (`stats.scores_path`) with the moderator.

Outputs under `derivatives/meeg-pipeline/group/stats/<analysis>/`:

- `*_clusters.tsv`: one row per cluster with p-value, time range, peak, and anatomical labels,
- `*_statistic.npz`: statistic map and null distribution,
- `*_stats.json`: settings, subjects used, and subjects skipped.

Existing results are kept unless the analysis name is listed in `OVERWRITE_ANALYSES`.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.stats import (
    make_stats_directory,
    run_cluster_analysis,
    stats_input_status_to_dataframe,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find the project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)

## Settings

Use a small number of permutations (for example 200) for a first look and `None` for the value from the config.

In [ ]:
ANALYSES = list(config.stats.analyses)   # or a list of names, for example ["h1", "h4"]
N_PERMUTATIONS = None                    # None: use stats.n_permutations
OVERWRITE_ANALYSES = []                  # analyses whose existing results are replaced

pd.DataFrame(
    [
        {
            "analysis": name,
            "kind": config.stats.analyses[name].kind,
            "terms": len(config.stats.analyses[name].terms),
            "moderator": config.stats.analyses[name].moderator,
            "overwrite": name in OVERWRITE_ANALYSES,
        }
        for name in ANALYSES
    ]
)

## Input status

One row per analysis and subject. Subjects with missing inputs are skipped by the analysis and listed in the JSON sidecar.

In [ ]:
status = stats_input_status_to_dataframe(config)
status = status[status["analysis"].isin(ANALYSES)]

display(
    status.groupby(["analysis", "status"]).size().unstack(fill_value=0)
)
status[status["status"] != "ready"]

## Run the tests

The permutation tests can take several minutes per analysis.

In [ ]:
results = []
for name in ANALYSES:
    result = run_cluster_analysis(
        config,
        name,
        n_permutations=N_PERMUTATIONS,
        on_existing="overwrite" if name in OVERWRITE_ANALYSES else "skip",
    )
    results.append(result)

pd.DataFrame([vars(result) for result in results])

## Clusters

Significant clusters have a p-value below `stats.alpha`.

In [ ]:
for name in ANALYSES:
    table_path = make_stats_directory(config, name) / f"desc-{name}_clusters.tsv"
    if not table_path.exists():
        print(f"{name}: no cluster table yet")
        continue

    table = pd.read_csv(table_path, sep="\t")
    print(f"{name}: {int(table['significant'].sum())} significant of {len(table)} clusters")
    display(table[table["significant"]].head(20))

## Null distribution

Observed cluster statistics against the permutation distribution of the largest cluster.

In [ ]:
fig, axes = plt.subplots(1, len(ANALYSES), figsize=(4 * len(ANALYSES), 3), squeeze=False)

for ax, name in zip(axes[0], ANALYSES):
    path = make_stats_directory(config, name) / f"desc-{name}_statistic.npz"
    if not path.exists():
        ax.set_title(f"{name}: no result")
        continue

    data = np.load(path, allow_pickle=True)
    null = data["null_distribution"]
    observed = np.abs(data["cluster_stats"])
    ax.hist(null, bins=40, color="0.7")
    if observed.size:
        ax.axvline(np.percentile(null, 100 * (1 - config.stats.alpha)), color="k", linestyle="--")
        for value in observed[observed > 0][:5]:
            ax.axvline(value, color="C3")
    ax.set_title(name)
    ax.set_xlabel("max |cluster statistic|")

fig.tight_layout()